# Visualização 3D — Interlagos + gNB
Execute **Run All** (Shift+Enter em cada célula) para ver o preview interativo.

In [1]:
import os, pickle, math
import numpy as np
import sionna.rt as rt

os.chdir('/mnt/stg/tls/blender')

# Carrega configuração da gNB
with open('output/gnb_config.pkl', 'rb') as f:
    cfg = pickle.load(f)
pos_gnb = cfg['pos_enu']
print(f'gNB ENU: E={pos_gnb[0]:.2f} m, N={pos_gnb[1]:.2f} m, Z={pos_gnb[2]:.1f} m')

gNB ENU: E=-7.54 m, N=-2.89 m, Z=25.0 m


In [2]:
# Carrega e configura cena
scene = rt.load_scene('output/interlagos.xml')
scene.frequency = cfg['freq_hz']
scene.bandwidth = cfg['bw_hz']

scene.tx_array = rt.PlanarArray(
    num_rows=cfg['num_rows'], num_cols=cfg['num_cols'],
    vertical_spacing=cfg['spacing'], horizontal_spacing=cfg['spacing'],
    pattern=cfg['pattern'], polarization=cfg['polariz'],
)
scene.rx_array = rt.PlanarArray(num_rows=1, num_cols=1, pattern='iso', polarization='V')

scene.add(rt.Transmitter(name='gnb', position=pos_gnb, orientation=cfg['orientacao']))
scene.add(rt.Receiver(name='rx_test', position=[100., 0., 1.5]))
print('Cena pronta.')

Cena pronta.


In [3]:
# RadioMap rápido para overlay (cell_size=10m, 500k samples)
rm_solver = rt.RadioMapSolver()
rm = rm_solver(
    scene=scene,
    center=[-76., 1.5, 126.],
    orientation=[0., 0., math.pi / 2],
    size=[1880., 1860.],
    cell_size=[10., 10.],
    samples_per_tx=500_000,
    max_depth=3,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    diffraction=True,
)
print('RadioMap calculado.')

RadioMap calculado.


In [4]:
# Preview 3D interativo
# - Orbitar: arrastar com o mouse
# - Zoom: scroll
# - Clique em objeto: mostra nome e propriedades
scene.preview(
    radio_map=rm,
    rm_metric='path_gain',
    show_devices=True,
    show_orientations=True,
    resolution=(1280, 720),
    fov=60.0,
)

In [5]:
# Opcional: PathSolver para visualizar raios na cena
solver = rt.PathSolver()
paths = solver(
    scene=scene,
    max_depth=3,
    los=True,
    specular_reflection=True,
    diffraction=True,
    diffuse_reflection=False,
    synthetic_array=True,
)
valid = np.array(paths.valid)
print(f'Caminhos válidos: {int(valid.sum())}')

# Preview com raios
scene.preview(
    paths=paths,
    show_devices=True,
    resolution=(1280, 720),
    fov=60.0,
)

Caminhos válidos: 3
